# PF2e Spell Metadata - первая PyTorch multi-label модель

Модель принимает очищенное описание заклинания и его rank, а затем отдельно предсказывает `keywords` и официальные `traits`. По умолчанию inference возвращает не больше 5 keywords и 5 traits; эти значения можно менять без переобучения.

In [8]:
# Этап 1. Импорты и конфигурация.
# Комментарий после этапа: все параметры собраны в одном месте, чтобы запуск
# можно было повторить и чтобы вторая модель могла использовать тот же контракт.
from __future__ import annotations

import json
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd()
DATA_DIR = ROOT / "spellsdata"
ARTIFACT_DIR = ROOT / "artifacts" / "first_model"
SEED = 42
MAX_TEXT_LENGTH = 256
MAX_VOCAB_SIZE = 12000
MIN_KEYWORD_DOCUMENT_FREQUENCY = 3
DEFAULT_TOP_K_KEYWORDS = 5
DEFAULT_TOP_K_TRAITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}, data_dir={DATA_DIR}")

device=cpu, data_dir=d:\fontys\semester3\PF2e-splells-creations\spellsdata


## Этап 2. Загрузка и подготовка targets

Для traits используются официальные значения Foundry из `system.traits.value` и traditions. Keywords строятся из текста как воспроизводимые pseudo-labels: берутся содержательные токены с минимальной document frequency. Это отдельная задача от официальных traits и не смешивает свободные слова с VTT-метаданными.

In [9]:
TOKEN_PATTERN = re.compile(r"[a-z][a-z0-9'-]{2,}")
HTML_PATTERN = re.compile(r"<[^>]+>")
UUID_PATTERN = re.compile(r"@UUID\[[^]]+\](?:\{([^}]+)\})?")
STOP_WORDS = {
    "the", "and", "for", "that", "this", "with", "from", "into", "your", "you",
    "are", "can", "each", "when", "then", "than", "has", "have", "will", "its",
    "not", "one", "only", "once", "all", "any", "use", "used", "using", "spell",
}

def clean_text(value: str) -> str:
    value = HTML_PATTERN.sub(" ", value or "")
    value = UUID_PATTERN.sub(lambda match: match.group(1) or " ", value)
    return " ".join(value.lower().split())

def tokenize(value: str) -> list[str]:
    return [token for token in TOKEN_PATTERN.findall(clean_text(value)) if token not in STOP_WORDS]

def load_spells(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(data_dir.rglob("*.json")):
        if path.name == "_folders.json":
            continue
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            system = payload.get("system", {})
            description = system.get("description", {}).get("value", "")
            traits_data = system.get("traits", {})
            traits = traits_data.get("value", []) or []
            traditions = traits_data.get("traditions", []) or []
            text = clean_text(description)
            labels = sorted({str(label).casefold() for label in [*traits, *traditions] if label})
            if text and labels:
                rows.append({"name": payload.get("name", path.stem), "text": text,
                             "tokens": tokenize(text), "rank": int(system.get("level", {}).get("value", 0) or 0),
                             "traits": labels})
        except (OSError, json.JSONDecodeError, TypeError, ValueError) as error:
            print(f"Skipped {path}: {error}")
    return pd.DataFrame(rows)

df = load_spells(DATA_DIR)
document_frequency = Counter({token for tokens in df["tokens"] for token in set(tokens)})
keyword_labels = sorted(token for token, count in document_frequency.items()
                        if count >= MIN_KEYWORD_DOCUMENT_FREQUENCY)
keyword_set = set(keyword_labels)
df["keywords"] = df["tokens"].apply(lambda tokens: sorted(set(tokens) & keyword_set))
df = df[df["keywords"].map(bool)].reset_index(drop=True)
trait_labels = sorted({trait for labels in df["traits"] for trait in labels})

print(f"usable spells={len(df)}")
print(f"keyword labels={len(keyword_labels)}, trait labels={len(trait_labels)}")
print(df[["name", "rank", "keywords", "traits"]].head(3).to_string(index=False))

usable spells=0
keyword labels=0, trait labels=0
Empty DataFrame
Columns: [name, rank, keywords, traits]
Index: []


## Этап 3. Vocabulary, split и DataLoader

Текст превращается в `input_ids`, rank передаётся отдельным числом, а две группы labels - в multi-hot vectors. Split фиксирован seed; vocabulary и `pos_weight` будут построены только по train-части, чтобы не использовать test information.

In [ ]:
def split_frame(frame: pd.DataFrame, seed: int = SEED) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    test_start = int(len(shuffled) * 0.8)
    valid_start = int(len(shuffled) * 0.9)
    return shuffled.iloc[:test_start], shuffled.iloc[test_start:valid_start], shuffled.iloc[valid_start:]

train_df, valid_df, test_df = split_frame(df)
train_token_frequency = Counter(token for tokens in train_df["tokens"] for token in tokens)
vocabulary_tokens = [token for token, _ in train_token_frequency.most_common(MAX_VOCAB_SIZE - 2)]
token_to_id = {"<PAD>": 0, "<UNK>": 1, **{token: index + 2 for index, token in enumerate(vocabulary_tokens)}}
trait_to_id = {label: index for index, label in enumerate(trait_labels)}
keyword_to_id = {label: index for index, label in enumerate(keyword_labels)}

def multi_hot(labels: list[str], mapping: dict[str, int]) -> Tensor:
    vector = torch.zeros(len(mapping), dtype=torch.float32)
    for label in labels:
        if label in mapping:
            vector[mapping[label]] = 1.0
    return vector

class SpellDataset(Dataset[dict[str, Tensor]]):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, Tensor]:
        row = self.frame.iloc[index]
        ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in row["tokens"][:MAX_TEXT_LENGTH]]
        ids += [token_to_id["<PAD>"]] * (MAX_TEXT_LENGTH - len(ids))
        return {"input_ids": torch.tensor(ids, dtype=torch.long), "rank": torch.tensor([row["rank"] / 10.0], dtype=torch.float32),
                "keywords": multi_hot(row["keywords"], keyword_to_id), "traits": multi_hot(row["traits"], trait_to_id)}

train_loader = DataLoader(SpellDataset(train_df), batch_size=32, shuffle=True)
valid_loader = DataLoader(SpellDataset(valid_df), batch_size=64)
test_loader = DataLoader(SpellDataset(test_df), batch_size=64)

def calculate_pos_weight(frame: pd.DataFrame, labels: list[str], column: str) -> Tensor:
    positives = torch.tensor([sum(label in values for values in frame[column]) for label in labels], dtype=torch.float32)
    return ((len(frame) - positives) / positives.clamp_min(1)).clamp(max=20.0)

keyword_pos_weight = calculate_pos_weight(train_df, keyword_labels, "keywords").to(DEVICE)
trait_pos_weight = calculate_pos_weight(train_df, trait_labels, "traits").to(DEVICE)
print(f"split sizes: train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

## Этап 4. Архитектура первой модели

Используется Embedding и двунаправленный GRU для контекста текста. Rank объединяется с pooled text representation перед общим MLP. Две головы возвращают logits без sigmoid: это необходимо для численно стабильного `BCEWithLogitsLoss`.

In [ ]:
class SpellTagger(nn.Module):
    def __init__(self, vocab_size: int, keyword_count: int, trait_count: int, embedding_dim: int = 96, hidden_dim: int = 96) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.encoder = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.shared = nn.Sequential(nn.Linear(hidden_dim * 2 + 1, 128), nn.ReLU(), nn.Dropout(0.25))
        self.keywords_head = nn.Linear(128, keyword_count)
        self.traits_head = nn.Linear(128, trait_count)

    def forward(self, input_ids: Tensor, rank: Tensor) -> dict[str, Tensor]:
        embedded = self.embedding(input_ids)
        encoded, _ = self.encoder(embedded)
        mask = input_ids.ne(0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        shared = self.shared(torch.cat([pooled, rank], dim=1))
        return {"keywords": self.keywords_head(shared), "traits": self.traits_head(shared)}

model = SpellTagger(len(token_to_id), len(keyword_to_id), len(trait_to_id)).to(DEVICE)
losses = {"keywords": nn.BCEWithLogitsLoss(pos_weight=keyword_pos_weight), "traits": nn.BCEWithLogitsLoss(pos_weight=trait_pos_weight)}
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
print(f"parameters={sum(parameter.numel() for parameter in model.parameters()):,}")

## Этап 5. Обучение и multi-label decoding

Сначала обучаем обе головы совместно. Затем decoder применяет threshold и сортирует labels по вероятности, ограничивая результат отдельным `top_k`. Поэтому модель не обязана всегда возвращать ровно одинаковое число labels.

In [ ]:
def move_batch(batch: dict[str, Tensor]) -> dict[str, Tensor]:
    return {key: value.to(DEVICE) for key, value in batch.items()}

def run_epoch(loader: DataLoader[dict[str, Tensor]], training: bool) -> float:
    model.train(training)
    total_loss = 0.0
    for batch in loader:
        batch = move_batch(batch)
        with torch.set_grad_enabled(training):
            logits = model(batch["input_ids"], batch["rank"])
            loss = losses["keywords"](logits["keywords"], batch["keywords"]) + losses["traits"](logits["traits"], batch["traits"])
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        total_loss += loss.item() * batch["input_ids"].size(0)
    return total_loss / len(loader.dataset)

history: list[dict[str, float]] = []
best_valid_loss = float("inf")
for epoch in range(1, 16):
    train_loss = run_epoch(train_loader, training=True)
    valid_loss = run_epoch(valid_loader, training=False)
    history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), ARTIFACT_DIR / "model.pt")
    print(f"epoch={epoch:02d} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f}")

model.load_state_dict(torch.load(ARTIFACT_DIR / "model.pt", map_location=DEVICE, weights_only=True))

@torch.no_grad()
def predict_probabilities(text: str, rank: int) -> dict[str, Tensor]:
    tokens = tokenize(text)[:MAX_TEXT_LENGTH]
    ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in tokens]
    ids += [token_to_id["<PAD>"]] * (MAX_TEXT_LENGTH - len(ids))
    batch = {"input_ids": torch.tensor([ids], dtype=torch.long, device=DEVICE),
             "rank": torch.tensor([[rank / 10.0]], dtype=torch.float32, device=DEVICE)}
    logits = model(batch["input_ids"], batch["rank"])
    return {key: value.sigmoid().squeeze(0).cpu() for key, value in logits.items()}

def decode(probabilities: Tensor, labels: list[str], threshold: float, top_k: int) -> list[dict[str, float | str]]:
    candidates = [(labels[index], float(probability)) for index, probability in enumerate(probabilities) if probability >= threshold]
    candidates.sort(key=lambda item: item[1], reverse=True)
    return [{"label": label, "probability": round(probability, 4)} for label, probability in candidates[:max(0, top_k)]]

def predict_spell(text: str, rank: int, top_k_keywords: int = DEFAULT_TOP_K_KEYWORDS,
                  top_k_traits: int = DEFAULT_TOP_K_TRAITS, keyword_threshold: float = 0.35,
                  trait_threshold: float = 0.35) -> dict[str, Any]:
    probabilities = predict_probabilities(text, rank)
    return {"text": text, "rank": rank,
            "keywords": decode(probabilities["keywords"], keyword_labels, keyword_threshold, top_k_keywords),
            "traits": decode(probabilities["traits"], trait_labels, trait_threshold, top_k_traits),
            "probabilities": {key: values.tolist() for key, values in probabilities.items()},
            "model_version": "first-pytorch-multilabel-v1"}


## Этап 6. Оценка и сохранение preprocessing artifacts

Оценка использует test только после обучения. Macro-F1 показывает качество по редким labels, micro-F1 - общее качество по всем решениям. Порог по умолчанию пока явный и настраиваемый; следующий шаг может подобрать его на validation отдельно для каждой головы.

In [ ]:
@torch.no_grad()
def collect_predictions(loader: DataLoader[dict[str, Tensor]]) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    model.eval()
    probabilities: dict[str, list[np.ndarray]] = {"keywords": [], "traits": []}
    targets: dict[str, list[np.ndarray]] = {"keywords": [], "traits": []}
    for raw_batch in loader:
        batch = move_batch(raw_batch)
        outputs = model(batch["input_ids"], batch["rank"])
        for key in probabilities:
            probabilities[key].append(outputs[key].sigmoid().cpu().numpy())
            targets[key].append(batch[key].cpu().numpy())
    return {key: np.concatenate(value) for key, value in probabilities.items()}, {key: np.concatenate(value) for key, value in targets.items()}

def multilabel_metrics(probabilities: np.ndarray, targets: np.ndarray, threshold: float = 0.35) -> dict[str, float]:
    predicted = probabilities >= threshold
    true_positive = np.logical_and(predicted, targets == 1).sum()
    false_positive = np.logical_and(predicted, targets == 0).sum()
    false_negative = np.logical_and(~predicted, targets == 1).sum()
    micro_precision = true_positive / max(true_positive + false_positive, 1)
    micro_recall = true_positive / max(true_positive + false_negative, 1)
    micro_f1 = 2 * micro_precision * micro_recall / max(micro_precision + micro_recall, 1e-8)
    per_label_f1: list[float] = []
    for column in range(targets.shape[1]):
        tp = np.logical_and(predicted[:, column], targets[:, column] == 1).sum()
        fp = np.logical_and(predicted[:, column], targets[:, column] == 0).sum()
        fn = np.logical_and(~predicted[:, column], targets[:, column] == 1).sum()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        per_label_f1.append(2 * precision * recall / max(precision + recall, 1e-8))
    return {"micro_f1": float(micro_f1), "macro_f1": float(np.mean(per_label_f1)),
            "subset_accuracy": float(np.all(predicted == targets, axis=1).mean())}

test_probabilities, test_targets = collect_predictions(test_loader)
metrics = {key: multilabel_metrics(test_probabilities[key], test_targets[key]) for key in ("keywords", "traits")}
print(json.dumps(metrics, indent=2))

metadata = {"token_to_id": token_to_id, "keyword_labels": keyword_labels, "trait_labels": trait_labels,
            "max_text_length": MAX_TEXT_LENGTH, "default_top_k_keywords": DEFAULT_TOP_K_KEYWORDS,
            "default_top_k_traits": DEFAULT_TOP_K_TRAITS, "seed": SEED, "device_at_training": str(DEVICE)}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"saved artifacts to {ARTIFACT_DIR}")

## Этап 7. Проверка handoff для второй модели

Следующая модель сможет принимать этот объект без повторного NLP parsing: в нём есть rank, выбранные labels, их probabilities и версия первой модели. Полные вероятности оставлены для co-occurrence/post-processing.

In [ ]:
example_prediction = predict_spell(
    "You channel a burst of molten fire in a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage.",
    rank=3,
    top_k_keywords=5,
    top_k_traits=5,
)
print(json.dumps({key: value for key, value in example_prediction.items() if key != "probabilities"}, indent=2))
# Комментарий после этапа: первая модель завершает только candidate generation.
# Вторая модель сможет использовать полный probability vector и co-occurrence,
# а top-k останется пользовательской настройкой inference.
